# Timestral (zero-shot) vs AutoETS

**Question.** Does Mistral's pretrained time-series model Timestral, used zero-shot with the weather as known-future covariates, beat the univariate AutoETS model from `forecast.ipynb` on 48-hour generation forecasts?

**Short answer** (skill = RMSE / seasonal-naive RMSE, lower is better):
- **Cross-validation:** TODO after the run.
- **Final holdout:** TODO after the run.
- **Intervals:** TODO after the run (native quantiles vs CV-calibrated).
- **Weather:** TODO after the run (Section 6).

**Models**

| Model | Inputs | Context |
|---|---|---|
| Timestral + weather\* | `y`, 7 weather columns as known-future covariates | 180 days |
| Timestral (no covariates) | `y` only | 180 days |
| AutoETS | `log(y)`, additive error/trend/season, 24-hour season | 46 days |
| Seasonal naive | last observed day, repeated twice | — |

Timestral is not trained here: the same pretrained weights (`mistralai/Timestral-209M`) forecast every window from its context alone. There is nothing to fit, so "training window" becomes "context length".

\* **Optimistic weather.** Historical evaluation feeds the *observed* weather from `data/train.csv` over the forecast horizon, not the forecast that would have been available. Section 6.3 estimates how much accuracy this hides.

**Contents**
1. Data
2. Evaluation design: rolling origins, CV and holdout
3. Timestral setup
4. Cross-validation
5. Prediction intervals and the final holdout
6. What drives Timestral
7. Forecast for the next 48 hours
8. Summary and conclusions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tbaraniuk/time-series_energy-consumption/blob/main/notebooks/timestral_baseline.ipynb)

**Locally:** run `uv sync` and open from the project root or `notebooks/`. **On Colab:** pick a GPU runtime (*Runtime → Change runtime type*); the setup cell below clones the repository and installs the missing packages. Helpers live in `src/final/timestral_baseline.py` and `src/final/lgbm_baseline.py`. The first run downloads about 840 MB of weights from Hugging Face.

In [ ]:
# Colab only: fetch the repository (data + helpers) and install what Colab lacks.
import os
import subprocess
import sys

if 'google.colab' in sys.modules:
    REPO_URL = 'https://github.com/tbaraniuk/time-series_energy-consumption.git'
    REPO_DIR = '/content/time-series_energy-consumption'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsforecast',
                    'timestral @ git+https://github.com/mistralai/timestral.git'], check=True)
    # pyproject.toml pins Python >= 3.14, which Colab does not have; import src/ directly instead.
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

In [ ]:
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='IProgress not found')
warnings.filterwarnings('ignore', category=FutureWarning)

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from final.lgbm_baseline import (
    DAY, DAY_PARTS, H, PARTS, WEATHER_COLUMNS,
    autoets, calibrate_halfwidths, interval, mae, plot_window_grid, rmse, seasonal_naive, winkler,
)
from final.timestral_baseline import MODEL_COLORS, MODEL_ID, MODEL_NAMES, load_timestral, timestral_forecast

plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
pd.set_option('display.precision', 2)

ROOT = Path.cwd() if (Path.cwd() / 'data/train.csv').exists() else Path.cwd().parent
CONTEXT_DAYS = 180          # Same history length as the LightGBM training window.
ETS_TRAIN_DAYS = 46         # Same rolling AutoETS window as forecast.ipynb.
CV_N_FOLDS = 7
HOLDOUT_DAYS = 14
CONTEXT_GRID_DAYS = [14, 46, 90, 180, 365, 730]
NOISE_REPEATS = 5
WEATHER_NOISE_LEVELS = [0, 0.25, 0.5, 1.0]
SEED = 42

## 1. Data

Hourly generation `y` with seven weather columns; see `lightgbm_baseline.ipynb` (Section 1) for the exploratory plots. The checks below make sure the history has no gaps and that the weather forecast covers exactly the next 48 hours.

In [ ]:
data = pd.read_csv(ROOT / 'data/train.csv', parse_dates=['ds']).sort_values('ds').reset_index(drop=True)
future_weather = (
    pd.read_csv(ROOT / 'data/weather_forecast.csv', parse_dates=['ds'])
    .sort_values('ds')
    .reset_index(drop=True)
)

assert data['ds'].diff().iloc[1:].eq(pd.Timedelta(hours=1)).all(), 'History must be hourly without gaps.'
expected_future = pd.date_range(data['ds'].iloc[-1] + pd.Timedelta(hours=1), periods=H, freq='h')
assert pd.DatetimeIndex(future_weather['ds']).equals(expected_future), 'Weather forecast must cover the next 48 hours.'
assert (data['y'] > 0).all(), 'log-ETS needs positive values.'

y = data['y'].to_numpy(dtype=float)
daylight = data['daylight'].to_numpy(dtype=int)
N = len(y)

print(f'{N:,} hourly observations: {data.ds.iloc[0]:%Y-%m-%d} to {data.ds.iloc[-1]:%Y-%m-%d}')
print(f'Forecast target: {future_weather.ds.iloc[0]:%Y-%m-%d %H:%M} to {future_weather.ds.iloc[-1]:%Y-%m-%d %H:%M}')

### 1.1 The level is not stable

Generation stayed near 200 for two years, then roughly quadrupled during 2026 and fell again in October. A long context mixes several regimes; Section 6.2 checks how much context Timestral actually wants.

In [ ]:
series = data.set_index('ds').y
fig, ax = plt.subplots(figsize=(12, 4))
series.resample('7D').mean().plot(ax=ax, color='grey', alpha=0.7, label='7-day mean')
series.resample('30D').mean().plot(ax=ax, color='black', lw=2, label='30-day mean')
ax.axvspan(data.ds.iloc[-CONTEXT_DAYS * DAY], data.ds.iloc[-1], color=MODEL_COLORS['timestral'], alpha=0.08,
           label=f'Last {CONTEXT_DAYS} days (Timestral context)')
ax.set(title='Generation level over time', ylabel='y', xlabel='')
ax.legend(loc='upper left')
plt.show()

## 2. Evaluation design

Identical to `lightgbm_baseline.ipynb`, so the scores are directly comparable. Every forecast is made from an **origin**: the model sees only data before it and predicts the next 48 hours.

- **Cross-validation (7 origins, 2 days apart):** used to compare models and to calibrate prediction intervals.
- **Holdout (last 14 days, 7 origins):** touched once, with all settings frozen.

At each origin Timestral reads the previous 180 days and AutoETS is refit on the previous 46 days. Seven CV windows are few: treat differences between models smaller than the fold-to-fold spread as noise.

In [ ]:
holdout_start = N - HOLDOUT_DAYS * DAY
holdout_origins = np.arange(holdout_start, N - H + 1, H, dtype=int)
cv_origins = holdout_start - H - np.arange(CV_N_FOLDS - 1, -1, -1) * H

if int(cv_origins[0]) - max(CONTEXT_GRID_DAYS) * DAY < 0:
    raise ValueError('Not enough history for the longest context in CONTEXT_GRID_DAYS.')

stages = [('CV', o) for o in cv_origins] + [('Holdout', o) for o in holdout_origins]
fig, ax = plt.subplots(figsize=(12, 5))
for row, (stage, origin) in enumerate(stages):
    start = mdates.date2num(data.ds.iloc[origin])
    ax.barh(row, CONTEXT_DAYS, left=start - CONTEXT_DAYS, height=0.7,
            color=MODEL_COLORS['timestral'], alpha=0.2, label='Timestral context (180 d)' if row == 0 else None)
    ax.barh(row, ETS_TRAIN_DAYS, left=start - ETS_TRAIN_DAYS, height=0.35,
            color=MODEL_COLORS['autoets'], alpha=0.6, label='AutoETS training (46 d)' if row == 0 else None)
    ax.barh(row, H / DAY, left=start, height=0.7, color='tab:orange' if stage == 'CV' else 'tab:red',
            label={0: 'CV window (48 h)', CV_N_FOLDS: 'Holdout window (48 h)'}.get(row))

left_edge = mdates.date2num(data.ds.iloc[cv_origins[0]]) - 55
ax.set_xlim(left_edge, mdates.date2num(data.ds.iloc[-1]) + 3)
ax.text(left_edge + 1, -0.9, '← Timestral context continues 180 days back', fontsize=8, color='grey')
ax.set_yticks(range(len(stages)), [f'{stage} {i % CV_N_FOLDS + 1}' for i, (stage, _) in enumerate(stages)])
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.set_title('Rolling origins: every row is one forecast')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=8, frameon=False)
plt.show()

## 3. Timestral setup

- **Target context:** `y` over the last 180 days before the origin, as one univariate series (shape `(origins, 1, time)`). All origins of a stage go through the model in one batch.
- **Covariates:** the seven weather columns are passed as *known-future* covariates over context + horizon. Timestral treats 0/1 rows (`daylight`) as indicators. Covariates condition the forecast; they are never forecast themselves.
- **Output:** 21 quantiles per hour. The median is the point forecast; the 2.5% and 97.5% quantiles (interpolated between the 1% and 5% levels, and between 95% and 99%) give a native 95% interval.
- No scaling or log transform: the model normalises each context internally. Negative quantiles are clipped at 0.

In [ ]:
model = load_timestral(MODEL_ID)
cache = {}


def timestral(origins, key='timestral', context_days=CONTEXT_DAYS, covariates=WEATHER_COLUMNS, target_weather=None):
    """{origin: (median, lower, upper)}; cached unless the horizon weather is replaced."""
    if key == 'timestral_uni':
        covariates = []
    cache_key = (key, context_days, tuple(covariates), tuple(int(o) for o in origins))
    if target_weather is not None:
        return timestral_forecast(model, data, origins, context_days, covariates, target_weather)
    if cache_key not in cache:
        cache[cache_key] = timestral_forecast(model, data, origins, context_days, covariates)
    return cache[cache_key]


ets_cache = {}


def predict_all(origins):
    """{model_key: {origin: point}} plus Timestral's native intervals {key: {origin: (lower, upper)}}."""
    points, native = {}, {}
    for key in ('timestral', 'timestral_uni'):
        result = timestral(origins, key)
        points[key] = {o: result[o][0] for o in origins}
        native[key] = {o: result[o][1:] for o in origins}
    for o in origins:
        if o not in ets_cache:
            ets_cache[o] = autoets(data, o, ETS_TRAIN_DAYS)
    points['autoets'] = {o: ets_cache[o] for o in origins}
    points['naive'] = {o: seasonal_naive(y, o) for o in origins}
    return points, native

## 4. Cross-validation

All four models forecast the same seven CV windows. The main score is **skill = RMSE / seasonal-naive RMSE** (below 1 means better than naive). Tables use the ratio of mean RMSEs, so one unusually easy window for naive cannot dominate the average. MAE is shown alongside RMSE. MAPE is not used: night values close to zero make it explode.

In [ ]:
def score_rows(origin, model_key, point, lower=None, upper=None, interval_kind=None):
    actual = y[origin : origin + H]
    naive = seasonal_naive(y, origin)
    rows = []
    for part, sl in PARTS.items():
        row = {
            'origin': int(origin),
            'model_key': model_key,
            'model': MODEL_NAMES[model_key],
            'part': part,
            'RMSE': rmse(actual[sl], point[sl]),
            'MAE': mae(actual[sl], point[sl]),
            'RMSE / naive': rmse(actual[sl], point[sl]) / rmse(actual[sl], naive[sl]),
        }
        if lower is not None:
            inside = (actual[sl] >= lower[sl]) & (actual[sl] <= upper[sl])
            row.update({
                'interval': interval_kind,
                'coverage': inside.mean(),
                'width': np.mean(upper[sl] - lower[sl]),
                'Winkler': winkler(actual[sl], lower[sl], upper[sl]),
            })
        rows.append(row)
    return rows


def summarize(scores, metrics):
    """Mean of each metric per model and part; 'RMSE / naive' is the ratio of mean RMSEs."""
    values = sorted({m for m in metrics if m != 'RMSE / naive'} | {'RMSE'})
    table = scores.pivot_table(index='model', columns='part', values=values, aggfunc='mean')
    if 'RMSE / naive' in metrics:
        skill = table['RMSE'] / table.loc[MODEL_NAMES['naive'], 'RMSE']
        table = pd.concat([table, pd.concat({'RMSE / naive': skill}, axis=1)], axis=1)
    return table[metrics].reindex(index=list(MODEL_NAMES.values())).reindex(columns=list(PARTS), level=1)


cv_predictions, cv_native = predict_all(cv_origins)
cv_scores = pd.DataFrame([
    row for key in MODEL_NAMES for o in cv_origins for row in score_rows(o, key, cv_predictions[key][o])
])

print('Mean over 7 CV windows')
display(summarize(cv_scores, ['RMSE / naive', 'RMSE', 'MAE']).round(2))

### 4.1 How stable is the ranking across windows?

One dot per CV window; dashed lines are means.

In [ ]:
compared = ['timestral', 'timestral_uni', 'autoets']
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
fold_numbers = np.arange(1, CV_N_FOLDS + 1)
for ax, part in zip(axes, DAY_PARTS):
    for offset, key in zip([-0.18, 0, 0.18], compared):
        values = cv_scores.query('model_key == @key and part == @part').set_index('origin')['RMSE / naive']
        values = values.reindex(cv_origins).to_numpy()
        ax.scatter(fold_numbers + offset, values, color=MODEL_COLORS[key], s=40, label=MODEL_NAMES[key], zorder=3)
        ax.axhline(values.mean(), color=MODEL_COLORS[key], ls='--', lw=1)
    ax.axhline(1, color=MODEL_COLORS['naive'], lw=1.5, label='Seasonal naive = 1')
    ax.set(title=f'Hours {part}', xlabel='CV window', xticks=fold_numbers)
axes[0].set_ylabel('RMSE / seasonal naive (lower is better)')
axes[0].legend(fontsize=8)
fig.suptitle('Per-window skill; dashed lines are means')
plt.tight_layout()
plt.show()

### 4.2 Every CV window

Grey: the last observed day. Shaded: the hidden 48 hours. The dotted line separates forecast day 1 from day 2. Titles show RMSE / naive over all 48 hours. Hourly values for each window are in `cv_tables[origin]`.

In [ ]:
def window_title(scores, origin, prefix):
    skill = scores.query('origin == @origin and part == "1–48"').set_index('model_key')['RMSE / naive']
    return (f'{prefix} · {data.ds.iloc[origin]:%b %d} — Timestral {skill["timestral"]:.2f}, '
            f'AutoETS {skill["autoets"]:.2f}')


fig, axes = plt.subplots(4, 2, figsize=(14, 13), sharey=True)
plot_window_grid(
    fig, axes, data.ds, y, cv_origins,
    {o: {k: cv_predictions[k][o] for k in MODEL_NAMES} for o in cv_origins},
    [window_title(cv_scores, o, f'CV {i}') for i, o in enumerate(cv_origins, start=1)],
    names=MODEL_NAMES, colors=MODEL_COLORS,
)
fig.suptitle('Cross-validation windows: actual vs forecasts', y=1.0)
plt.tight_layout()
plt.show()

cv_tables = {
    o: pd.DataFrame({'ds': data.ds.iloc[o : o + H].to_numpy(), 'actual': y[o : o + H]}
                    | {MODEL_NAMES[k]: cv_predictions[k][o] for k in MODEL_NAMES})
    for o in cv_origins
}

## 5. Prediction intervals and the final holdout

Two kinds of 95% interval:

- **Native:** Timestral's own 2.5% and 97.5% quantiles. No calibration data needed.
- **Calibrated:** as in `lightgbm_baseline.ipynb`: the 95th percentile of absolute CV errors, split by forecast day and the `daylight` flag, used as a symmetric half-width around the point forecast. Every model gets this one, which keeps the comparison fair.

Calibrated intervals rest on only 7 × 48 errors per model, so neither kind is a coverage guarantee.

In [ ]:
halfwidths = {key: calibrate_halfwidths(y, daylight, cv_origins, cv_predictions[key]) for key in MODEL_NAMES}
display(
    pd.DataFrame(halfwidths).T.rename(index=MODEL_NAMES)
    .rename(columns=lambda flag: 'day' if flag else 'night', level=1)
    .round(0)
)

**Holdout.** The same models and interval widths, frozen, on the last 14 days. Coverage should be near 0.95. The Winkler score combines width with a penalty for every miss (lower is better).

In [ ]:
holdout_predictions, holdout_native = predict_all(holdout_origins)
holdout_forecasts = {}
holdout_rows = []
for origin in holdout_origins:
    holdout_forecasts[origin] = {}
    for key in MODEL_NAMES:
        point = holdout_predictions[key][origin]
        lower, upper = interval(point, daylight[origin : origin + H], halfwidths[key])
        holdout_forecasts[origin][key] = (point, lower, upper)
        holdout_rows += score_rows(origin, key, point, lower, upper, interval_kind='calibrated')
        if key in holdout_native:
            holdout_rows += score_rows(origin, key, point, *holdout_native[key][origin], interval_kind='native')
holdout_scores = pd.DataFrame(holdout_rows)
calibrated = holdout_scores.query('interval == "calibrated"')

print('Mean over 7 holdout windows')
display(summarize(calibrated, ['RMSE / naive', 'RMSE', 'MAE']).round(2))
print('Calibrated intervals')
display(summarize(calibrated, ['coverage', 'width', 'Winkler']).round(2))
print('Timestral native intervals')
display(summarize(holdout_scores.query('interval == "native"'), ['coverage', 'width', 'Winkler']).dropna().round(2))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 13), sharey=True)
plot_window_grid(
    fig, axes, data.ds, y, holdout_origins,
    {o: {k: holdout_forecasts[o][k][0] for k in MODEL_NAMES} for o in holdout_origins},
    [window_title(calibrated, o, f'Holdout {i}') for i, o in enumerate(holdout_origins, start=1)],
    bands={o: ('timestral', *holdout_native['timestral'][o]) for o in holdout_origins},
    names=MODEL_NAMES, colors=MODEL_COLORS,
)
fig.suptitle('Holdout windows with the native Timestral 95% interval', y=1.0)
plt.tight_layout()
plt.show()

### 5.1 Which windows and which forecast day?

Each cell is RMSE / naive for one holdout window. Blue is better than naive, red is worse.

In [ ]:
skill = (
    calibrated.query('model_key != "naive" and part != "1–48"')
    .pivot_table(index='origin', columns=['model_key', 'part'], values='RMSE / naive')
    .reindex(columns=[(k, p) for k in compared for p in DAY_PARTS])
)
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(skill.to_numpy(), cmap='RdBu_r', vmin=0.4, vmax=1.6, aspect='auto')
ax.grid(False)
ax.set_xticks(range(skill.shape[1]), [f'{MODEL_NAMES[k]}\nhours {p}' for k, p in skill.columns], fontsize=8)
ax.set_yticks(range(len(skill)), [f'{data.ds.iloc[o]:%b %d}' for o in skill.index])
for i in range(skill.shape[0]):
    for j in range(skill.shape[1]):
        ax.text(j, i, f'{skill.iat[i, j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='RMSE / naive')
ax.set_title('Holdout skill by window')
plt.tight_layout()
plt.show()

### 5.2 Error by forecast hour and interval quality

Left: RMSE for each of the 48 steps, averaged over holdout windows. Right: each point is one model, interval kind and forecast day; the best interval is close to the dashed 0.95 line with the smallest width. Hollow markers are Timestral's native intervals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
steps = np.arange(1, H + 1)
for key in MODEL_NAMES:
    errors = np.stack([y[o : o + H] - holdout_forecasts[o][key][0] for o in holdout_origins])
    axes[0].plot(steps, np.sqrt(np.mean(errors ** 2, axis=0)), color=MODEL_COLORS[key], label=MODEL_NAMES[key])
axes[0].axvline(DAY + 0.5, color='grey', ls=':', lw=1)
axes[0].set(title='Holdout RMSE by forecast hour', xlabel='Forecast hour', ylabel='RMSE')
axes[0].legend(fontsize=8)

for kind, fill in [('calibrated', True), ('native', False)]:
    table = summarize(holdout_scores.query('interval == @kind'), ['coverage', 'width'])
    for key, name in MODEL_NAMES.items():
        if table.loc[name].isna().all():
            continue
        for part, marker in zip(DAY_PARTS, ['o', 's']):
            x, c = table.loc[name, ('width', part)], table.loc[name, ('coverage', part)]
            axes[1].scatter(x, c, marker=marker, s=70, color=MODEL_COLORS[key] if fill else 'none',
                            edgecolors=MODEL_COLORS[key], linewidths=1.5)
            axes[1].annotate(f'{part}', (x, c), xytext=(6, 4), textcoords='offset points', fontsize=8)
axes[1].axhline(0.95, color='black', ls='--', lw=1)
axes[1].set(title='Interval coverage vs width (holdout)', xlabel='Mean width', ylabel='Coverage')
handles = [plt.Line2D([], [], color=MODEL_COLORS[k], marker='o', ls='', label=n) for k, n in MODEL_NAMES.items()]
handles.append(plt.Line2D([], [], color='grey', marker='o', mfc='none', ls='', label='native (hollow)'))
axes[1].legend(handles=handles, fontsize=8, loc='center right')
axes[1].margins(0.15)
plt.tight_layout()
plt.show()

## 6. What drives Timestral

Timestral has no feature importances, but its inputs can still be varied: which covariates it sees, how much history it reads, and how accurate the weather is. All experiments use the seven CV windows.

### 6.1 Removing a covariate group

Forecast again without one weather group (the full model and the no-covariate model are the two extremes). Bars are mean RMSE increase over the seven windows, whiskers are standard errors (SD / √7). A whisker that crosses zero means the effect is not distinguishable from window-to-window noise.

In [ ]:
COVARIATE_GROUPS = {
    'Sun and clouds': ['cloud_cover', 'daylight'],
    'Wind': ['wind_speed', 'air_density'],
    'Other weather': ['temperature', 'pressure', 'humidity'],
    'All weather': WEATHER_COLUMNS,
}
GROUP_COLORS = {'Sun and clouds': '#ff7f0e', 'Wind': '#17becf', 'Other weather': '#e377c2', 'All weather': '#7f7f7f'}
assert sorted(sum(list(COVARIATE_GROUPS.values())[:-1], [])) == sorted(WEATHER_COLUMNS)

ablation_rows = []
for group, removed in COVARIATE_GROUPS.items():
    kept = [c for c in WEATHER_COLUMNS if c not in removed]
    reduced = timestral(cv_origins, covariates=kept)
    for origin in cv_origins:
        actual, full = y[origin : origin + H], cv_predictions['timestral'][origin]
        for part in DAY_PARTS:
            sl = PARTS[part]
            ablation_rows.append({'group': group, 'part': part,
                                  'delta_RMSE': rmse(actual[sl], reduced[origin][0][sl]) - rmse(actual[sl], full[sl])})

ablation = pd.DataFrame(ablation_rows).groupby(['group', 'part']).delta_RMSE.agg(['mean', 'sem'])
group_order = ablation['mean'].groupby('group').mean().sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for ax, part in zip(axes, DAY_PARTS):
    sub = ablation.xs(part, level='part').reindex(group_order)
    ax.barh(sub.index, sub['mean'], xerr=sub['sem'], capsize=3, color=[GROUP_COLORS[g] for g in sub.index])
    ax.axvline(0, color='black', lw=1)
    ax.set(title=f'Hours {part}', xlabel='RMSE increase after removing the group')
fig.suptitle('Covariate ablation (forecast again without each group)')
plt.tight_layout()
plt.show()

### 6.2 How much history should Timestral read?

The level shifts in Section 1.1 suggest that a long context could mislead the model, while a short one hides the yearly pattern. Pooled CV skill for several context lengths, with and without weather. The 180-day setting used everywhere else was fixed before looking at this plot; do not pick a new value from it and then reuse the same CV windows to score it.

In [ ]:
naive_rmse = {
    part: rmse(np.concatenate([y[o : o + H][PARTS[part]] for o in cv_origins]),
               np.concatenate([seasonal_naive(y, o)[PARTS[part]] for o in cv_origins]))
    for part in DAY_PARTS
}


def pooled_skill(forecasts):
    """Pooled RMSE / naive per forecast day for {origin: (median, ...)}."""
    return {
        part: rmse(np.concatenate([y[o : o + H][PARTS[part]] for o in cv_origins]),
                   np.concatenate([forecasts[o][0][PARTS[part]] for o in cv_origins])) / naive_rmse[part]
        for part in DAY_PARTS
    }


context_rows = []
for days in CONTEXT_GRID_DAYS:
    for key in ('timestral', 'timestral_uni'):
        for part, value in pooled_skill(timestral(cv_origins, key, context_days=days)).items():
            context_rows.append({'days': days, 'model_key': key, 'part': part, 'skill': value})
context = pd.DataFrame(context_rows)

fig, ax = plt.subplots(figsize=(8, 4))
for key in ('timestral', 'timestral_uni'):
    for part, ls in zip(DAY_PARTS, ['-', '--']):
        sub = context.query('model_key == @key and part == @part')
        ax.plot(sub.days, sub.skill, ls=ls, marker='o', color=MODEL_COLORS[key], label=f'{MODEL_NAMES[key]}, hours {part}')
ax.axhline(1, color=MODEL_COLORS['naive'], lw=1.5, label='Seasonal naive')
ax.axvline(CONTEXT_DAYS, color='grey', ls=':', lw=1)
ax.set_xscale('log')
ax.set_xticks(CONTEXT_GRID_DAYS, [str(d) for d in CONTEXT_GRID_DAYS])
ax.set(xlabel='Context length (days, log scale)', ylabel='Pooled CV RMSE / naive', title='Skill vs context length')
ax.legend(fontsize=7)
plt.show()

### 6.3 How much does perfect weather flatter Timestral?

Historical scores use observed weather over the horizon. Here Gaussian noise is added to the horizon weather of the CV windows (the context keeps the true history), with standard deviation equal to a fraction of each column's spread over the last 180 days. `daylight` is left untouched because it is known exactly in advance.

This is a rough stress test, not a model of real forecast errors, but it shows how much of the advantage survives imperfect weather. The no-covariate model is the floor: once noisy weather is worse than no weather, the covariates are doing harm.

In [ ]:
noisy_columns = [c for c in WEATHER_COLUMNS if c != 'daylight']
column_sd = data[noisy_columns].iloc[-CONTEXT_DAYS * DAY:].std().to_numpy()
rng = np.random.default_rng(SEED)

noise_rows = []
for level in WEATHER_NOISE_LEVELS:
    for repeat in range(NOISE_REPEATS if level else 1):
        noisy = {}
        for o in cv_origins:
            frame = data.iloc[o : o + H].copy()
            frame[noisy_columns] += rng.normal(0, 1, (H, len(noisy_columns))) * level * column_sd
            noisy[o] = frame
        for part, value in pooled_skill(timestral(cv_origins, target_weather=noisy)).items():
            noise_rows.append({'level': level, 'part': part, 'skill': value})
noise = pd.DataFrame(noise_rows).groupby(['level', 'part']).skill.agg(['mean', 'std']).fillna(0).unstack('part')
uni_skill = pooled_skill(timestral(cv_origins, 'timestral_uni'))

fig, ax = plt.subplots(figsize=(8, 4))
for part, ls in zip(DAY_PARTS, ['-', '--']):
    ax.errorbar(noise.index, noise[('mean', part)], yerr=noise[('std', part)], ls=ls, marker='o',
                color=MODEL_COLORS['timestral'], capsize=3, label=f'Timestral + weather, hours {part}')
    ax.axhline(uni_skill[part], color=MODEL_COLORS['timestral_uni'], ls=ls, lw=1.2,
               label=f'Timestral no covariates, hours {part}')
ax.axhline(1, color=MODEL_COLORS['naive'], lw=1.5, label='Seasonal naive')
ax.set(xlabel='Weather noise (fraction of column SD)', ylabel='Pooled CV RMSE / naive',
       title='Timestral skill when weather inputs are noisy')
ax.legend(fontsize=7)
plt.show()

## 7. Forecast for the next 48 hours

Timestral reads the last 180 days of `data/train.csv` and the weather forecast from `data/weather_forecast.csv` as the horizon covariates. AutoETS is refit at the last observed hour. This is a preview only: no submission file is written.

In [ ]:
future_daylight = future_weather.daylight.to_numpy(dtype=int)
future_forecasts = {}
for key in ('timestral', 'timestral_uni'):
    covariates = WEATHER_COLUMNS if key == 'timestral' else []
    future_forecasts[key] = timestral_forecast(model, data, [N], CONTEXT_DAYS, covariates, {N: future_weather})[N]
for key, point in [('autoets', autoets(data, N, ETS_TRAIN_DAYS)), ('naive', seasonal_naive(y, N))]:
    future_forecasts[key] = (point, *interval(point, future_daylight, halfwidths[key]))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(data.ds.iloc[-4 * DAY:], y[-4 * DAY:], color='black', lw=1.2, label='History')
point, lower, upper = future_forecasts['timestral']
ax.fill_between(future_weather.ds, lower, upper, color=MODEL_COLORS['timestral'], alpha=0.12, lw=0,
                label='95% interval: Timestral (native)')
for key, (point, _, _) in future_forecasts.items():
    ax.plot(future_weather.ds, point, color=MODEL_COLORS[key], label=MODEL_NAMES[key])
ax.axvline(future_weather.ds.iloc[0], color='grey', ls='--', lw=1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.set(title='Forecast for the next 48 hours', ylabel='y', xlabel='')
ax.set_ylim(bottom=0)
ax.legend(ncol=2, fontsize=8)
plt.show()

forecast_preview = pd.DataFrame({'ds': future_weather.ds} | {MODEL_NAMES[k]: f[0] for k, f in future_forecasts.items()})

## 8. Summary and conclusions

Skill is RMSE / seasonal naive (below 1 is better). Holdout interval columns are averaged over all 48 hours.

In [ ]:
summary = pd.concat({
    'CV skill': summarize(cv_scores, ['RMSE / naive'])['RMSE / naive'],
    'Holdout skill': summarize(calibrated, ['RMSE / naive'])['RMSE / naive'],
} | {
    f'Holdout {kind} interval': summarize(holdout_scores.query('interval == @kind'), ['coverage', 'Winkler'])
    .xs('1–48', axis=1, level='part')
    for kind in ('calibrated', 'native')
}, axis=1)
display(summary.round(2))

**Caveats**
- Timestral + weather scores use observed horizon weather (Section 6.3 shows the sensitivity to weather errors).
- Seven windows per stage are few; the fold-to-fold spread in Section 4.1 can be as large as the gaps between models.
- The holdout falls on a sharply declining level (Section 1.1), which favours models that react fast to the last day.
- Timestral is zero-shot: it was pretrained on other series and never fitted here. A fine-tuned model could do better, but would need its own validation.

**Conclusions**

TODO: write 3–5 conclusions from the summary table and the plots above.